# 01. 프롬프트 엔지니어링 실습

---

## 학습 목표

LLM의 출력 품질을 높이는 **핵심 프롬프트 기법 6가지**를 실습합니다.  
같은 질문에 다른 기법을 적용해 결과 차이를 직접 확인합니다.

## 다루는 기법

| 기법 | 핵심 아이디어 | 언제 쓰나? |
|---|---|---|
| **Zero-shot** | 예시 없이 바로 질문 | 간단한 일반 질문 |
| **Few-shot** | 예시(샷)를 프롬프트에 포함 | 특정 형식·스타일 유도 |
| **Chain-of-Thought** | 단계별 추론 유도 | 복잡한 분석·판단 |
| **역할 프롬프트** | LLM에게 페르소나 부여 | 전문 도메인 답변 |
| **구조화된 출력** | 특정 형식(JSON 등) 강제 | 파싱이 필요한 경우 |
| **프롬프트 템플릿** | 변수로 재사용 가능한 프롬프트 | 반복 작업 자동화 |

## 흐름

```
기법 학습 → 코드 실습 → 같은 질문으로 비교 실험 → 핵심 정리
```

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import json

In [16]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# import json

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("준비 완료")
print("실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'")

준비 완료
실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'


---

## 1. Zero-shot Prompting

**예시 없이** 바로 질문하는 가장 기본적인 방식입니다.  
LLM이 사전 학습 지식만으로 답변합니다.

```
사용자: [질문]
LLM:   [답변]
```

In [17]:
# ── Zero-shot: 예시 없이 바로 질문 ────────────────────────────
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
zero_shot_prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [18]:
# 체인 생성 및 실행
chain = zero_shot_prompt | llm | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-openai': '1.3.4'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool

In [19]:
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"}) # prompt | llm | parser

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
재택근무의 전면 도입에 대한 찬반 의견은 여러 가지 측면에서 고려할 수 있습니다. 찬성하는 이유와 반대하는 이유를 각각 살펴보겠습니다.

### 찬성하는 이유

1. **유연한 근무 환경**: 재택근무는 직원들에게 유연한 근무 시간을 제공하여 개인의 생활과 일의 균형을 맞출 수 있게 합니다.

2. **교통비 및 시간 절약**: 출퇴근 시간을 줄일 수 있어 직원들이 더 많은 시간을 개인적인 활동이나 가족과 함께 보낼 수 있습니다.

3. **비용 절감**: 기업 측면에서 사무실 운영 비용(임대료, 관리비 등)을 줄일 수 있습니다.

4. **생산성 향상**: 일부 연구에 따르면 재택근무가 직원의 생산성을 높일 수 있다는 결과도 있습니다. 조용한 환경에서 집중할 수 있기 때문입니다.

5. **인재 확보**: 지역에 구애받지 않고 인재를 채용할 수 있어 다양한 인재를 확보할 수 있습니다.

### 반대하는 이유

1. **소통의 어려움**: 팀원 간의 즉각적인 소통이 어려워질 수 있으며, 이는 협업에 부정적인 영향을 미칠 수 있습니다.

2. **업무 집중도 저하**: 가정에서의 다양한 방해 요소(가사, 가족 등)로 인해 업무 집중도가 떨어질 수 있습니다.

3. **조직 문화 약화**: 사무실에서의 대면 소통이 줄어들면 조직 문화가 약화될 수 있으며, 팀워크가 저하될 수 있습니다.

4. **경계의 모호함**: 일과 개인 생활의 경계가 모호해져, 직원들이 과중한 업무에 시달릴 수 있습니다.

5. **기술적 문제**: 재택근무를 위한 기술적 인프라가 부족한 경우, 업무 효율성이 떨어질 수 있습니다.

결론적으로, 재택근무의 전면 도입은 기업의 특성과 직원들의 선호도에 따라 다르게 평가될 수 있습니다. 각 기업은 이러한 장단점을 고려하여 최적의 근무 방식을 선택하는 것이 중요합니다.


In [20]:
# 제로샷 테스트
# 질문을 작성 > 체인 실행 > 결과 출력
result = chain.invoke({"question": "자율 주행 트럭 전면 도입, 찬성해야 할까요?"}) # prompt | llm | parser

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
자율 주행 트럭의 전면 도입에 대한 찬반 의견은 여러 가지 측면에서 고려할 수 있습니다. 찬성하는 이유와 반대하는 이유를 각각 살펴보겠습니다.

### 찬성하는 이유

1. **안전성 향상**: 자율 주행 기술은 인간의 실수를 줄여 교통사고를 감소시킬 수 있습니다. 특히 피로, 주의 산만 등으로 인한 사고를 예방할 수 있습니다.

2. **효율성 증가**: 자율 주행 트럭은 최적의 경로를 계산하고 연료 효율성을 극대화할 수 있어 물류 비용을 절감할 수 있습니다.

3. **24시간 운영 가능**: 자율 주행 트럭은 휴식이 필요 없기 때문에 24시간 운영이 가능하여 물류의 속도를 높일 수 있습니다.

4. **인력 부족 문제 해결**: 운전사 부족 문제를 해결할 수 있으며, 물류 산업의 지속적인 성장에 기여할 수 있습니다.

5. **환경적 이점**: 자율 주행 기술이 발전함에 따라 전기 트럭과 같은 친환경 차량의 도입이 촉진될 수 있습니다.

### 반대하는 이유

1. **일자리 감소**: 자율 주행 트럭의 도입은 운전사 일자리를 줄일 수 있으며, 이는 경제적 불평등을 심화시킬 수 있습니다.

2. **기술적 한계**: 현재 자율 주행 기술은 완벽하지 않으며, 복잡한 도로 상황이나 악천후에서의 안전성 문제가 여전히 존재합니다.

3. **법적 및 윤리적 문제**: 자율 주행 차량의 사고 발생 시 책임 소재가 불분명할 수 있으며, 윤리적 결정(예: 사고 회피 시 선택)과 관련된 문제도 있습니다.

4. **인프라 문제**: 자율 주행 트럭이 원활하게 운영되기 위해서는 도로 인프라와 통신 시스템이 개선되어야 하며, 이는 상당한 비용과 시간이 소요될 수 있습니다.

5. **사회적 수용성**: 사람들은 자율 주행 차량에 대한 신뢰가 부족할 수 있으며, 이는 도입에 대한 저항으로 이어질 수 있습니다.

결론적으로, 자율 주행 트럭의 전면 도입은 여러 장점과 단점을 가지고 있으며, 이를 종합적으로 고려하여 사회적 합의와 기술적 발

---

## 2. Few-shot Prompting

프롬프트 안에 **예시(shots)** 를 포함해서 원하는 출력 형식을 보여줍니다.  
LLM이 패턴을 학습해 같은 형식으로 답변합니다.

```
시스템: 예시1: Q→A, 예시2: Q→A  ← 패턴 학습
사용자: [새 질문]
LLM:   [예시와 같은 형식의 답변]  ← 패턴 따라함
```

In [22]:
# ── Few-shot: 예시 2개를 포함해서 출력 형식을 유도 ────────────
# TODO: 아래 예시 2개(질문→답변 형식)를 참고해 같은 형식으로 답하도록 지시하는
#       시스템 프롬프트를 작성하세요 (✅ 찬성 근거 / ❌ 반대 근거 / 💡 결론 형식)
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", 
    """ 아래의 예시 형식에 맞춰 답변하세요.

    질문 : "재택근무 전면 도입, 찬성해야 할까요?"
    
1. ✅ **찬성 근거**
   - 기업, 임직원, 사회적 측면의 주요 이점을 다룹니다 (예: 전국 단위 우수 인재 확보, 출퇴근 스트레스 감소 및 워라밸 향상, 고정비 절감 등).
   - 핵심 키워드에 볼드체를 사용하고 가독성 좋은 불렛 포인트로 정리합니다.

2. ❌ **반대 근거**
   - 운영, 조직 문화, 관리적 측면의 주요 한계와 문제점을 다룹니다 (예: 대면 소통 부재로 인한 협업 효율 저하, 신규 입사자 온보딩의 어려움, 조직 소속감 약화, 성과 관리의 어려움 등).
   - 핵심 키워드에 볼드체를 사용하고 가독성 좋은 불렛 포인트로 정리합니다.

3. 💡 **결론 및 제언**
   - 단순한 찬반 결론을 넘어서는 종합적인 통찰을 제공합니다.
   - 무조건적인 전면 도입보다는 현실적인 대안(예: 하이브리드 워크, 단계적 도입, 직무별 차등 적용)과 성공적인 정착을 위한 핵심 전제 조건을 제안합니다.
    """
    ),
    ("human", "질문: {question}")
])

In [23]:
# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template=' 아래의 예시 형식에 맞춰 답변하세요.\n\n    질문 : "재택근무 전면 도입, 찬성해야 할까요?"\n\n1. ✅ **찬성 근거**\n   - 기업, 임직원, 사회적 측면의 주요 이점을 다룹니다 (예: 전국 단위 우수 인재 확보, 출퇴근 스트레스 감소 및 워라밸 향상, 고정비 절감 등).\n   - 핵심 키워드에 볼드체를 사용하고 가독성 좋은 불렛 포인트로 정리합니다.\n\n2. ❌ **반대 근거**\n   - 운영, 조직 문화, 관리적 측면의 주요 한계와 문제점을 다룹니다 (예: 대면 소통 부재로 인한 협업 효율 저하, 신규 입사자 온보딩의 어려움, 조직 소속감 약화, 성과 관리의 어려움 등).\n   - 핵심 키워드에 볼드체를 사용하고 가독성 좋은 불렛 포인트로 정리합니다.\n\n3. 💡 **결론 및 제언**\n   - 단순한 찬반 결론을 넘어서는 종합적인 통찰을 제공합니다.\n   - 무조건적인 전면 도입보다는 현실적인 대안(예: 하이브리드 워크, 단계적 도입, 직무별 차등 적용)과 성공적인 정착을 위한 핵심 전제 조건을 제안합니다.\n    '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='질문: {question}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain

In [24]:
result = chain.invoke({"question": "주4일제 도입해야 할까?"}) # 문자열

print("=== Few-shot 결과 ===")
print(result)

=== Few-shot 결과 ===
1. ✅ **찬성 근거**
   - **생산성 향상**: 연구에 따르면, 주4일제로 근무하는 직원들이 더 높은 **생산성**을 보이는 경향이 있습니다.
   - **워라밸 개선**: 짧은 근무 시간으로 인해 **일과 삶의 균형**이 향상되어 직원들의 **정신적 건강**이 증진됩니다.
   - **인재 유치 및 유지**: 유연한 근무 환경은 **우수 인재**를 유치하고, 이직률을 낮추는 데 기여합니다.
   - **환경적 이점**: 출퇴근 일수가 줄어들어 **탄소 배출**이 감소하고, **교통 혼잡**이 완화됩니다.

2. ❌ **반대 근거**
   - **업무 효율성 저하**: 일부 산업에서는 **업무량**이 줄어들지 않아 직원들이 **과중한 업무**를 겪을 수 있습니다.
   - **고객 서비스 문제**: 고객과의 **소통**이 원활하지 않을 수 있으며, 서비스 제공에 **차질**이 생길 수 있습니다.
   - **조직 문화 변화의 어려움**: 기존의 **조직 문화**와 맞지 않아 직원들이 적응하는 데 어려움을 겪을 수 있습니다.
   - **성과 관리의 복잡성**: 주4일제로 인해 **성과 평가** 기준이 모호해질 수 있습니다.

3. 💡 **결론 및 제언**
   - 주4일제 도입은 **긍정적인 효과**를 가져올 수 있지만, 모든 산업과 조직에 일률적으로 적용하기에는 **한계**가 있습니다. 
   - **하이브리드 모델**을 고려하거나, **단계적 도입**을 통해 각 조직의 특성에 맞는 방안을 모색하는 것이 중요합니다.
   - 성공적인 정착을 위해서는 **명확한 성과 관리 기준**과 **조직 문화 변화**에 대한 준비가 필요합니다.


---

## 3. Chain-of-Thought (CoT) Prompting
생각의 고리

LLM에게 **단계별로 생각하도록** 유도합니다.  
복잡한 추론, 계획, 분석이 필요한 질문에 효과적입니다.

```
기본:  질문 → 바로 답변 (shallow)
CoT:  질문 → 단계1 → 단계2 → 단계3 → 결론 (deep)
```

In [25]:
# ── Chain-of-Thought: 단계별 사고 과정 명시 ───────────────────
# TODO: 4단계([1단계: 문제 파악]~[4단계: 최종 판단])로 생각하고
#       각 단계를 명시하도록 지시하는 시스템 프롬프트를 작성하세요
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """
        질문에 답변하기 전에 반드시 4단계로 생각하고, 각 단계를 명시적으로 출력하세요.
        [1단계: 문제 파악] 질문이 무엇인가?
        [2단계: 이해관계자 분석] 누가 영향을 받는가?
        [3단계: 장단점 분석] 적용했을 때 장점과 단점을 찾아서
        [4단계: 최종 판단] 근거를 제시하고 결론을 내린다.
    """),
    ("human", "{question}")
])

In [26]:
# 체인 생성 및 실행
chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Chain-of-Thought 결과 ===")
print(result)

=== Chain-of-Thought 결과 ===
[1단계: 문제 파악]  
질문은 재택근무를 전면 도입하는 것에 대한 찬성 여부입니다. 즉, 재택근무가 기업과 직원에게 긍정적인 영향을 미치는지, 또는 부정적인 영향을 미치는지를 평가해야 합니다.

[2단계: 이해관계자 분석]  
재택근무의 전면 도입은 여러 이해관계자에게 영향을 미칩니다.  
- **직원**: 재택근무를 통해 유연한 근무 환경을 누릴 수 있지만, 고립감이나 업무와 개인 생활의 경계가 모호해질 수 있습니다.  
- **기업**: 인건비 절감 및 생산성 향상을 기대할 수 있지만, 팀워크와 소통의 어려움이 발생할 수 있습니다.  
- **고객**: 서비스나 제품의 품질이 영향을 받을 수 있으며, 고객 지원의 효율성에 변화가 있을 수 있습니다.  
- **사회**: 교통 혼잡 감소와 환경 보호에 긍정적인 영향을 미칠 수 있지만, 지역 경제에 부정적인 영향을 미칠 수 있습니다.

[3단계: 장단점 분석]  
장점:  
- 유연한 근무 환경으로 직원의 만족도와 생산성이 증가할 수 있다.  
- 사무실 유지 비용 절감으로 기업의 재정적 부담이 줄어든다.  
- 교통 혼잡과 환경 오염 감소에 기여할 수 있다.  

단점:  
- 팀워크와 소통의 어려움으로 인해 협업이 저해될 수 있다.  
- 직원의 고립감이나 스트레스 증가로 인한 정신 건강 문제 발생 가능성이 있다.  
- 고객 서비스의 질이 저하될 수 있는 위험이 있다.

[4단계: 최종 판단]  
재택근무의 전면 도입은 장점과 단점이 모두 존재합니다. 그러나, 직원의 만족도와 생산성을 높이고, 기업의 비용을 절감할 수 있는 기회가 크기 때문에, 적절한 관리와 지원 시스템을 갖춘다면 찬성할 수 있습니다. 따라서, 재택근무를 전면 도입하는 것이 긍정적인 방향으로 나아갈 수 있다고 판단합니다.


---

## 4. 역할(Role) 프롬프트

LLM에게 **특정 전문가 페르소나**를 부여합니다.  
같은 질문이라도 역할에 따라 완전히 다른 관점으로 답변합니다.

```
역할 없음:  중립적·일반적 답변
HR 전문가:  사람 중심의 조직 관점
CFO:       비용·효율 중심의 재무 관점
```

In [27]:
question = "재택근무 전면 도입, 찬성해야 할까요?"

In [28]:
# ── 역할 프롬프트: 같은 질문, 다른 역할 ──────────────────────
def make_role_chain(role_description: str):
    """역할 설명을 받아 전문가 체인을 생성한다"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_description),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain

In [29]:
# TODO: 15년 경력 HR 디렉터 역할을 설명하는 문자열을 작성하세요
#       (직원 복지·조직 문화 중심, 3~4문장으로 답하도록 지시)
hr_chain = make_role_chain("""
                            구성원의 성장과 몰입을 최우선으로 두고, 성과와 사람의 조화를 이끄는 '사람 중심(People-Centric) HR 전문가'입니다.'
                            진정성 있고 공감하는 어조로 이야기하되, 조직의 지속 가능한 성장을 돕는 실질적이고 인간적인 HR 솔루션을 권고합니다.
                            """)

print("=== HR 디렉터 관점 ===")
print(hr_chain.invoke({"question": question}))

=== HR 디렉터 관점 ===
재택근무의 전면 도입에 대한 찬성 여부는 여러 가지 요소를 고려해야 합니다. 다음은 찬성할 수 있는 몇 가지 이유와 함께, 반대할 수 있는 요소도 함께 살펴보겠습니다.

### 찬성할 이유

1. **유연한 근무 환경**: 재택근무는 직원들에게 더 많은 유연성을 제공합니다. 개인의 생활 패턴에 맞춰 일할 수 있어, 일과 삶의 균형을 맞추는 데 도움이 됩니다.

2. **생산성 향상**: 많은 연구에서 재택근무가 직원의 생산성을 높일 수 있다는 결과가 나왔습니다. 통근 시간의 감소와 편안한 환경이 집중력을 높일 수 있습니다.

3. **비용 절감**: 기업은 사무실 운영 비용을 줄일 수 있으며, 직원들도 교통비와 식비 등의 비용을 절감할 수 있습니다.

4. **인재 확보 및 유지**: 재택근무는 더 넓은 인재 풀을 활용할 수 있게 해줍니다. 지역에 구애받지 않고 우수한 인재를 채용할 수 있는 기회를 제공합니다.

5. **환경적 이점**: 통근이 줄어들면서 탄소 배출량이 감소하고, 이는 환경 보호에 기여할 수 있습니다.

### 반대할 이유

1. **팀워크와 소통의 어려움**: 재택근무는 팀원 간의 즉각적인 소통을 어렵게 만들 수 있습니다. 이는 협업과 창의성에 부정적인 영향을 미칠 수 있습니다.

2. **업무와 개인 생활의 경계 모호**: 재택근무는 업무와 개인 생활의 경계를 흐리게 할 수 있어, 직원들이 과중한 업무에 시달리거나 번아웃을 경험할 수 있습니다.

3. **관리의 어려움**: 원격 근무 환경에서는 직원의 업무 진행 상황을 파악하기 어려울 수 있으며, 이는 관리자의 부담을 증가시킬 수 있습니다.

4. **기술적 문제**: 모든 직원이 원활한 재택근무를 위한 기술적 환경을 갖추고 있지 않을 수 있습니다. 이는 불평등을 초래할 수 있습니다.

5. **조직 문화의 약화**: 사무실에서의 대면 소통이 줄어들면서 조직 문화가 약화될 수 있습니다. 직원 간의 유대감이 감소할 위험이 있습니다.

### 결론

재택근무의

In [30]:
# TODO: 10년 경력 CFO 역할을 설명하는 문자열을 작성하세요
#       (비용-효익 분석 중심, 수치 근거로 3~4문장으로 답하도록 지시)
cfo_chain = make_role_chain("""
                            철저한 데이터와 재무 분석을 기반으로 조직의 수익성과 지속 가능성을 극대화하는 '전략적 CFO(최고재무책임자)'입니다.
                            숫자에 기반한 명확하고 논리적인 어조를 유지하며, 감정적 접근을 배제하고 단기와 장기적 재무 건전성을 보장하는 실용적 솔루션을 제안합니다.
                            """)

print("\n=== CFO 관점 ===")
print(cfo_chain.invoke({"question": question}))


=== CFO 관점 ===
재택근무 전면 도입에 대한 찬성 여부는 여러 가지 요소를 고려해야 합니다. 다음은 찬성할 수 있는 몇 가지 이유와 반대할 수 있는 이유를 정리한 것입니다.

### 찬성 이유

1. **비용 절감**: 사무실 운영 비용(임대료, 관리비 등)을 줄일 수 있습니다. 또한 직원들도 교통비와 식비를 절감할 수 있습니다.

2. **유연한 근무 환경**: 직원들이 자율적으로 근무 시간을 조정할 수 있어, 일과 삶의 균형을 맞추기 용이합니다. 이는 직원의 만족도와 생산성을 높일 수 있습니다.

3. **인재 확보**: 지리적 제약이 없으므로, 더 넓은 인재 풀에서 우수한 인재를 채용할 수 있습니다.

4. **재택근무의 효과성**: 많은 연구에서 재택근무가 생산성을 높일 수 있다는 결과가 나왔습니다. 특히 집중이 필요한 작업에 유리합니다.

### 반대 이유

1. **팀워크와 소통의 어려움**: 대면 소통이 줄어들면서 팀워크와 협업이 저해될 수 있습니다. 이는 창의성과 문제 해결 능력에 부정적인 영향을 미칠 수 있습니다.

2. **업무와 개인 생활의 경계 모호**: 재택근무는 업무와 개인 생활의 경계를 흐리게 할 수 있어, 직원들이 과중한 업무에 시달릴 위험이 있습니다.

3. **기술적 문제**: 원활한 재택근무를 위해서는 안정적인 인터넷과 적절한 기술적 지원이 필요합니다. 기술적 문제가 발생할 경우 업무에 차질이 생길 수 있습니다.

4. **관리의 어려움**: 직원의 업무 진행 상황을 모니터링하고 관리하는 것이 어려워질 수 있습니다. 이는 성과 평가와 피드백에 영향을 미칠 수 있습니다.

### 결론

재택근무의 전면 도입은 조직의 특성과 산업, 직원의 성향에 따라 다르게 평가될 수 있습니다. 따라서, 조직의 목표와 문화, 직원의 요구를 종합적으로 고려하여 결정하는 것이 중요합니다. 필요하다면 하이브리드 모델을 도입하여 유연성을 높이는 것도 하나의 대안이 될 수 있습니다.


In [ ]:
# 신입사원

new_chain = make_role_chain("""
                            조직에 새로운 활력을 불어넣고, 솔선수범하며 끊임없이 배우고 성장하려는 '열정적인 신입사원'입니다.
                            기존 방식에 호기심을 갖고 솔직하게 질문하며, 솔직하고 겸손하면서도 패기 있는 어조로 아이디어와 고민을 나눕니다.
                            """)
print("\n=== 신입사원 관점 ===")
print(new_chain.invoke({"question": question}))


=== 신입사원 관점 ===
재택근무 전면 도입에 대한 찬반 의견은 여러 가지가 있습니다. 찬성하는 이유와 반대하는 이유를 각각 살펴보면 다음과 같습니다.

### 찬성하는 이유:
1. **유연한 근무 환경**: 재택근무는 직원들에게 더 많은 유연성을 제공하여 개인의 생활과 업무를 조화롭게 할 수 있습니다.
2. **교통비 및 시간 절약**: 출퇴근 시간을 줄일 수 있어 직원들이 더 많은 시간을 개인적인 일이나 자기계발에 투자할 수 있습니다.
3. **생산성 향상**: 많은 연구에서 재택근무가 직원의 생산성을 높일 수 있다는 결과가 나왔습니다. 조용한 환경에서 집중할 수 있기 때문입니다.
4. **인재 유치 및 유지**: 재택근무를 제공하는 기업은 더 많은 인재를 유치할 수 있으며, 직원들의 이직률을 낮출 수 있습니다.

### 반대하는 이유:
1. **팀워크 및 소통의 어려움**: 대면 소통이 줄어들면서 팀워크와 협업이 어려워질 수 있습니다. 비언어적 소통이 부족해질 수 있습니다.
2. **업무와 개인 생활의 경계 모호**: 재택근무는 업무와 개인 생활의 경계를 흐리게 만들어, 직원들이 과중한 업무에 시달릴 수 있습니다.
3. **관리의 어려움**: 관리자가 직원의 업무 진행 상황을 파악하기 어려워질 수 있으며, 성과 평가가 복잡해질 수 있습니다.
4. **기술적 문제**: 재택근무를 위해 필요한 기술적 인프라가 부족한 경우, 업무에 차질이 생길 수 있습니다.

결론적으로, 재택근무의 전면 도입은 기업의 문화, 업무 성격, 직원들의 선호도에 따라 다르게 접근해야 할 문제입니다. 각 기업의 상황에 맞춰 유연한 정책을 마련하는 것이 중요합니다.


---

## 5. 구조화된 출력 (Structured Output)

LLM의 출력을 **JSON 등 특정 형식으로 강제**합니다.  
후처리(파싱, DB 저장, API 응답)가 필요할 때 사용합니다.

```
일반 출력: 자연어 문장 → 파싱 불가
구조화:   JSON/YAML → 자동 파싱 가능 → 시스템 연동 가능
```

In [36]:
# ── 구조화된 출력: JSON 형식 강제 ────────────────────────────
# TODO: topic·pros·cons·recommendation·confidence 필드를 가진 JSON으로만
#       답하도록(다른 텍스트 금지) 강제하는 시스템 프롬프트를 작성하세요
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        반드시 아래 형식으로 답변하세요. 다른 텍스트는 절대로 포함하면 안됩니다.
        
        - 출력형식: json

        "topic" : "주제",
        "pros" : ["찬성근거1", "찬성근거2", "찬성근거3"],
        "cons": ["반대근거1", "반대근거2"],
        "recommendation" : "최종 결론",
        "confidence" : "확신도(1~10)",
        
    """),
    ("human", "다음 주제를 분석해줘: {question}")
])

In [38]:
# 체인 생성 및 실행
chain = structured_prompt | llm | StrOutputParser()
raw_output = chain.invoke({"question": "재택근무 전면 도입"})

print("=== LLM 원본 출력 ===")
print(raw_output)

=== LLM 원본 출력 ===
{
    "topic": "재택근무 전면 도입",
    "pros": ["유연한 근무시간으로 직원의 워라밸 향상", "사무실 유지비용 절감", "지리적 제약 없이 인재 채용 가능"],
    "cons": ["팀워크와 소통의 어려움", "업무 집중도 저하 가능성", "보안 문제 발생 가능성"],
    "recommendation": "재택근무를 전면 도입하되, 팀 간 소통과 보안 문제를 해결할 수 있는 방안을 마련해야 한다.",
    "confidence": "8"
}


In [39]:
# JSON 파싱
print("\n=== 파싱 후 활용 ===")
try:
    data = json.loads(raw_output)
    print(f"주제     : {data['topic']}")
    print(f"확신도   : {data['confidence']}/10")
    print(f"권고사항 : {data['recommendation']}")
    print(f"찬성 근거: {', '.join(data['pros'])}")
except json.JSONDecodeError as e:
    print(f"파싱 실패: {e}")
    print("→ 팁: 모델이 JSON을 못 지키면 출력에서 JSON 부분만 추출하거나 재시도 로직을 추가하세요.")


=== 파싱 후 활용 ===
주제     : 재택근무 전면 도입
확신도   : 8/10
권고사항 : 재택근무를 전면 도입하되, 팀 간 소통과 보안 문제를 해결할 수 있는 방안을 마련해야 한다.
찬성 근거: 유연한 근무시간으로 직원의 워라밸 향상, 사무실 유지비용 절감, 지리적 제약 없이 인재 채용 가능


---

## 6. 프롬프트 템플릿 (재사용 가능한 팩토리)

**변수화된 프롬프트**를 만들어 다양한 도메인에 재사용합니다.  
같은 구조의 프롬프트를 반복해서 만들 때 효율적입니다.

```python
# 나쁜 예: 비슷한 프롬프트를 계속 복붙
legal_prompt   = "당신은 변호사입니다... {question}"
medical_prompt = "당신은 의사입니다... {question}"

# 좋은 예: 팩토리 함수로 생성
make_expert_chain(domain="법률", style="간결하게")
make_expert_chain(domain="의료", style="쉽게 설명")
```

In [40]:
# ── 프롬프트 템플릿 팩토리 ────────────────────────────────────
def make_analyst_chain(domain: str, perspective: str, output_format: str):
    """
    재사용 가능한 분석 체인을 생성한다.

    Args:
        domain      : 전문 도메인 (예: '경영', '기술', '심리')
        perspective : 분석 관점 (예: '비용 중심', '사람 중심')
        output_format: 출력 형식 지시문
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""당신은 {domain} 분야 전문 분석가입니다.
{perspective} 관점에서 분석하세요.

출력 형식:
{output_format}"""),
        ("human", "{question}")
    ])

    # 체인 생성
    chain = prompt | llm | StrOutputParser()
    return chain

In [ ]:
question = "재택근무 전면 도입, 찬성해야 할까요?"

In [43]:
# TODO: domain="경영", perspective="ROI와 생산성 중심"으로 채우고
#       output_format에 [핵심 지표]→[예상 효과]→[실행 권고] 형식을 지정하세요
biz_chain = make_analyst_chain(
    domain="경영",
    perspective="ROI와 생산성 중심",
    output_format="[핵심 지표]→[예상 효과]→[실행 권고]"
)

print("=== 경영 분석 관점 ===")
print(biz_chain.invoke({"question": question}))

=== 경영 분석 관점 ===
[핵심 지표]→[예상 효과]→[실행 권고]

[생산성] → [재택근무 도입으로 직원들이 자율적으로 일할 수 있는 환경이 조성되어, 업무 효율성이 증가할 것으로 예상됨. 또한, 통근 시간 절약으로 인해 직원들의 시간 관리가 개선되고, 집중력이 향상될 가능성이 높음.] → [재택근무의 효과를 극대화하기 위해, 명확한 목표 설정과 성과 측정을 위한 KPI를 도입하고, 정기적인 피드백 시스템을 구축할 것을 권장.]

[ROI] → [재택근무를 통해 사무실 운영 비용(임대료, 관리비 등)을 절감할 수 있으며, 인재 유치 및 유지에 긍정적인 영향을 미쳐 인적 자원에 대한 투자 수익률이 증가할 것으로 기대됨.] → [비용 절감 효과를 분석하고, 재택근무에 따른 인적 자원 관리 방안을 마련하여 장기적인 ROI를 극대화할 수 있는 전략을 수립할 것을 권장.]


In [44]:
# TODO: domain="조직심리", perspective="구성원 동기·몰입 중심"으로 채우고
#       output_format에 [심리적 영향]→[리스크]→[권고사항] 형식을 지정하세요
psych_chain = make_analyst_chain(
    domain="조직심리",
    perspective="구성원 동기·몰입 중심",
    output_format="[심리적 영향]→[리스크]→[권고사항]"
)

print("\n=== 조직심리 관점 ===")
print(psych_chain.invoke({"question": question}))


=== 조직심리 관점 ===
[심리적 영향]→재택근무는 구성원들에게 유연한 근무 환경을 제공하여 일과 삶의 균형을 개선할 수 있습니다. 이는 구성원들의 동기와 몰입을 높이는 긍정적인 요소로 작용할 수 있습니다. 그러나, 사회적 상호작용의 감소로 인해 고립감을 느끼거나 팀워크가 약화될 수 있는 부정적인 영향도 존재합니다.

[리스크]→재택근무가 장기화될 경우, 구성원 간의 소통 부족으로 인한 정보의 단절, 팀워크 저하, 그리고 직무에 대한 소속감 감소 등의 리스크가 발생할 수 있습니다. 이는 결국 조직의 성과와 혁신성에 부정적인 영향을 미칠 수 있습니다.

[권고사항]→재택근무를 도입할 경우, 정기적인 팀 미팅과 소통 채널을 활성화하여 구성원 간의 유대감을 유지하고, 팀워크를 강화하는 프로그램을 마련해야 합니다. 또한, 구성원들이 자율적으로 업무를 수행할 수 있도록 지원하되, 정기적인 피드백과 성과 평가를 통해 몰입을 지속적으로 유도하는 것이 중요합니다.


---

## 7. 비교 실험: 같은 질문, 다른 기법

지금까지 배운 6가지 기법을 **같은 질문**에 적용해 결과를 비교합니다.  
답변의 구조, 깊이, 형식이 얼마나 다른지 확인하세요.

In [45]:
# ── 비교 실험: 기법별 답변 길이 & 구조 비교 ──────────────────
QUESTION = "재택근무 전면 도입, 찬성해야 할까요?"

# 각 기법의 체인 목록
techniques = [
    ("Zero-shot",         zero_shot_prompt),
    ("Few-shot",          few_shot_prompt),
    ("Chain-of-Thought",  cot_prompt),
]

print(f"질문: {QUESTION}")
print("=" * 60)

results = {}
for name, prompt in techniques:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": QUESTION})
    results[name] = answer

# 결과 요약 출력
print(f"\n{'기법':<20} {'글자수':>8} {'줄 수':>6}")
print("-" * 38)
for name, answer in results.items():
    print(f"{name:<20} {len(answer):>8,} {len(answer.splitlines()):>6}")

질문: 재택근무 전면 도입, 찬성해야 할까요?

기법                        글자수    줄 수
--------------------------------------
Zero-shot                 946     27
Few-shot                  879     15
Chain-of-Thought          798     23


In [46]:
# ── 답변 전체 출력 ────────────────────────────────────────────
for name, answer in results.items():
    print(f"\n{'='*60}")
    print(f"[{name}]")
    print('='*60)
    # 처음 400자만 출력 (너무 길면 ...으로 줄임)
    preview = answer[:400]
    print(preview)
    if len(answer) > 400:
        print(f"\n... (총 {len(answer):,}자 중 400자 미리보기)")


[Zero-shot]
재택근무의 전면 도입에 대한 찬반 의견은 여러 가지 측면에서 고려할 수 있습니다. 찬성하는 이유와 반대하는 이유를 각각 살펴보겠습니다.

### 찬성하는 이유

1. **유연한 근무 환경**: 재택근무는 직원들에게 유연한 근무 시간을 제공하여 개인의 생활과 업무를 조화롭게 할 수 있습니다.

2. **교통비 및 시간 절약**: 출퇴근 시간을 줄일 수 있어 직원들이 더 많은 시간을 가족이나 개인적인 일에 사용할 수 있습니다.

3. **생산성 향상**: 일부 연구에 따르면, 재택근무가 직원의 생산성을 높일 수 있다는 결과도 있습니다. 조용한 환경에서 집중할 수 있는 기회가 많아지기 때문입니다.

4. **비용 절감**: 기업 측면에서도 사무실 운영 비용(임대료, 관리비 등)을 줄일 수 있습니다.

5. **인

... (총 946자 중 400자 미리보기)

[Few-shot]
1. ✅ **찬성 근거**
   - **우수 인재 확보**: 재택근무를 통해 전국 단위에서 우수한 인재를 모집할 수 있어, 기업의 경쟁력을 높일 수 있습니다.
   - **출퇴근 스트레스 감소**: 직원들이 출퇴근 시간을 절약함으로써 **워라밸**을 향상시키고, 전반적인 삶의 질을 높일 수 있습니다.
   - **고정비 절감**: 사무실 운영 비용이 줄어들어 기업의 **재무적 부담**이 경감됩니다. 
   - **유연한 근무 환경**: 직원들이 자신의 **업무 스타일**에 맞춰 일할 수 있어, 생산성이 증가할 가능성이 높습니다.

2. ❌ **반대 근거**
   - **협업 효율 저하**: 대면 소통이 줄어들어 **팀워크**와 협업의 효율성이 떨어질 수 있습니다.
   - **온보딩의 어려움**: 신규 입사

... (총 879자 중 400자 미리보기)

[Chain-of-Thought]
[1단계: 문제 파악]  
질문은 재택근무를 전면 도입하는 것에 대한 찬성 여부입니다. 즉, 재택근무가 기업과 직원에게 긍정적인 영향을 미치는지, 또는 부정적인 영향을 미치는지를 평

---

## 핵심 정리

```python
# 프롬프트 엔지니어링 선택 가이드

# 1. Zero-shot: 빠른 테스트, 일반 질문
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser()

# 2. Few-shot: 특정 형식 유도, 스타일 일관성
#    → 시스템 프롬프트에 예시 2~3개 포함

# 3. Chain-of-Thought: 복잡한 추론, 다단계 분석
#    → "단계별로 생각하세요" 지시문 추가

# 4. 역할 프롬프트: 전문 도메인, 특정 관점 필요 시
#    → 구체적일수록 좋음 ("전문가" < "10년 경력 변호사")

# 5. 구조화 출력: 파싱·연동이 필요한 경우
#    → JSON 형식 + "다른 텍스트 포함 금지" 명시

# 6. 팩토리 패턴: 동일 구조 반복 시
def make_chain(domain, style):
    prompt = ChatPromptTemplate.from_messages([...])
    return prompt | llm | StrOutputParser()
```

| 기법 | 최적 상황 | 주의사항 |
|---|---|---|
| Zero-shot | 범용 질문, 빠른 프로토타입 | 형식 보장 없음 |
| Few-shot | 특정 형식·스타일 필요 | 예시가 너무 많으면 토큰 낭비 |
| CoT | 복잡한 추론·판단 | 응답이 길어짐 |
| 역할 | 전문 도메인 | 구체적일수록 효과적 |
| 구조화 출력 | API 연동, 파싱 필요 | 100% 보장 안 됨 (검증 로직 필요) |
| 팩토리 | 반복 사용 패턴 | 과도한 추상화 주의 |
